# Learning on Graphs - Homework 1


The loading of Karate Club, matched Erdos-Renyi controls, Cora, the oversmoothing graph, and the bonus barbell graphs is provided for you.

In [1]:
%pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 843.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.9 MB/s eta 0:00:00


In [2]:
import os
import time
import random
import copy
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [3]:
# Provided graph/data loading.
G_karate = nx.karate_club_graph()
karate_club_labels = nx.get_node_attributes(G_karate, 'club')
karate_label_to_id = {'Mr. Hi': 0, 'Officer': 1}
karate_y = np.array([karate_label_to_id[karate_club_labels[i]]
                     for i in sorted(G_karate.nodes())])

n_karate = G_karate.number_of_nodes()
m_karate = G_karate.number_of_edges()
p_match = 2 * m_karate / (n_karate * (n_karate - 1))
er_graphs = [nx.erdos_renyi_graph(n_karate, p_match, seed=s) for s in [1, 2, 3]]

# Provided ER graph for the oversmoothing visualization in Part D.
def make_connected_er_graph(n=120, p=0.06, seed=7):
    for offset in range(1000):
        G = nx.erdos_renyi_graph(n, p, seed=seed + offset)
        if nx.is_connected(G):
            return G, seed + offset
    raise RuntimeError('Could not sample a connected ER graph')

G_over, G_over_seed = make_connected_er_graph(n=120, p=0.06, seed=7)
rng_over = np.random.default_rng(0)
over_signal0 = rng_over.normal(size=G_over.number_of_nodes())
over_signal0 = over_signal0 / np.max(np.abs(over_signal0))

# Provided graphs for the bonus.
G_barbell = nx.barbell_graph(8, 4)
G_shortcut = G_barbell.copy()
G_shortcut.add_edge(0, max(G_shortcut.nodes()))

# Cora citation network for Parts C and D.
dataset = Planetoid(root='data/Planetoid', name='Cora')
data = dataset[0].to(device)

print('Karate:', G_karate.number_of_nodes(), 'nodes,', G_karate.number_of_edges(), 'edges')
print('Karate labels in karate_y:', dict(zip(*np.unique(karate_y, return_counts=True))))
print('Matched ER p:', p_match)
print('Oversmoothing ER graph:', G_over.number_of_nodes(), 'nodes,', G_over.number_of_edges(), 'edges, seed', G_over_seed)
print(dataset)
print(data)
print('num_features:', dataset.num_features, 'num_classes:', dataset.num_classes)


Processing...


Karate: 34 nodes, 78 edges
Karate labels in karate_y: {np.int64(0): np.int64(17), np.int64(1): np.int64(17)}
Matched ER p: 0.13903743315508021
Oversmoothing ER graph: 120 nodes, 430 edges, seed 7
Cora()
Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
num_features: 1433 num_classes: 7


Done!


## Implementation note

The notebook loads the graphs and datasets for you, but it does **not** implement the graph-operator utilities. You should implement graph statistics, adjacency/Laplacian construction, normalized GCN matrices, Dirichlet energy, and pairwise-distance diagnostics yourself in the cells below. In particular, please note that you are expected to implement the operators yourselves, rather than using functions from pytorch-geometric or other libraries.

In [ ]:
# TODO: implement these helper functions. Do not import them from another file.
# Keep the function names fixed so later cells are easy to write and grade.

def graph_stats(G, name='graph'):
    """Return graph statistics: nodes, edges, average degree, density, components, min degree, max degree."""
    # TODO: implement using NetworkX and NumPy.
    raise NotImplementedError


def adjacency_matrix_np(G):
    """Return the dense adjacency matrix using sorted node order."""
    # TODO: implement using NetworkX.
    raise NotImplementedError


def laplacian_np(A):
    """Return L = D - A."""
    # TODO: implement using NumPy.
    raise NotImplementedError


def normalized_adjacency_np(A, add_self_loops=True):
    """Return D_hat^{-1/2} A_hat D_hat^{-1/2}, with A_hat = A + I when add_self_loops=True."""
    # TODO: implement the symmetric GCN normalization.
    raise NotImplementedError


def symmetric_gcn_matrix_nx(G):
    """Return S and L for a NetworkX graph, where S = D_hat^{-1/2} A_hat D_hat^{-1/2}."""
    # TODO: construct A_hat, degrees, S, and L.
    raise NotImplementedError


def symmetric_gcn_matrix_from_edge_index(edge_index, num_nodes, device):
    """Return S and L for a PyG graph represented by edge_index."""
    # TODO: construct a dense adjacency from edge_index, add self-loops, then compute S and L.
    raise NotImplementedError


def dirichlet_energy(H, L):
    """Return Tr(H^T L H) as a Python float."""
    # TODO: implement with torch operations.
    raise NotImplementedError


def mean_pairwise_distance(H, max_nodes=512):
    """Return the mean pairwise Euclidean distance between node embeddings.
    For large graphs, subsample at most max_nodes nodes.
    """
    # TODO: implement using torch.cdist.
    raise NotImplementedError


## Part B: NetworkX graph operators

In [ ]:
# Provided visual overview.
pos = nx.spring_layout(G_karate, seed=0)
node_colors = karate_y
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
nx.draw_networkx(G_karate, pos=pos, node_color=node_colors, cmap='tab10',
                 with_labels=True, node_size=320, font_size=8, ax=axes[0])
axes[0].set_title('Karate Club colored by club')
axes[0].axis('off')
nx.draw_networkx(er_graphs[0], pos=nx.spring_layout(er_graphs[0], seed=1),
                 with_labels=False, node_size=120, width=0.7, ax=axes[1])
axes[1].set_title('One matched ER example')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# B1 TODO: report Karate statistics and club counts.
# Use karate_club_labels and karate_y from the loading cell.

In [ ]:
# B2 TODO: compare Karate with all three ER graphs and plot the degree histograms.

In [ ]:
# B3 TODO: build A, D, L, and A_norm for Karate and report shape/nnz/density.

In [ ]:
# B4 TODO:
# Use karate_y from the loading cell.
# Define signal0 = 1 - 2 * karate_y.
# Smooth it with A_norm and visualize k=0,1,5,20 using vmin=-1 and vmax=1 in every panel.

## Part C: Semi-supervised node classification with GCN on Cora

In [ ]:
# C1 TODO: report Cora dataset statistics and explain why this setup is transductive.

In [ ]:
# C2-C4 TODO: implement exactly these two models: MLP and two-layer GCN.
# Train both for exactly 500 epochs and select the checkpoint with best validation accuracy.

# TODO: implement count_parameters
# TODO: implement accuracy_from_logits
# TODO: implement MLP
# TODO: implement GCN using GCNConv
# TODO: implement train_pyg_model(..., epochs=500, ...)
# TODO: train MLP and GCN, then plot loss and validation accuracy over 500 epochs

## Part D: Depth and oversmoothing with fixed GCN propagation

In this part there are **no trainable weights**. Use the fixed GCN propagation matrix
\[
S = \widehat D^{-1/2}\widehat A\widehat D^{-1/2}, \quad \widehat A=A+I.
\]
Starting from Cora node features, compute \(H^{(k+1)} = S H^{(k)}\). This isolates the smoothing effect of repeated GCN propagation.


In [ ]:
# D1-D2 TODO: fixed GCN propagation on Cora, no training.
# Build S_cora and L_cora with symmetric_gcn_matrix_from_edge_index.
# Starting from H = data.x.float(), compute H_{k+1} = S_cora H_k for depths [0,1,2,4,8,16,32,64].
# For each listed depth, report Dirichlet energy and mean pairwise distance.
# Plot both diagnostics versus propagation depth.

In [ ]:
# D3 TODO: use the provided ER graph G_over and signal over_signal0.
# Propagate with the symmetric GCN matrix for depths [0,1,2,4,8,16,32,64].
# Plot Dirichlet energy, variance, and mean pairwise distance.
# Visualize all snapshots with the same color range vmin=-1 and vmax=1.

In [ ]:
# D4 TODO: short reflection.
# Explain why repeated fixed GCN propagation improves communication range but can make node representations less discriminative.

## Part E: 1-WL and expressivity toy study

Path5 and Star5 are a same-size warmup pair with 5 nodes each. The real failure example is C6 versus 2K3, which are both 6-node graphs. Do not compare 5-node graphs to 6-node graphs as an isomorphism test.


In [ ]:
# Provided toy graph visualization.
graphs = {
    'Path5': nx.path_graph(5),
    'Star5': nx.star_graph(4),
    'C6': nx.cycle_graph(6),
    '2K3': nx.disjoint_union(nx.complete_graph(3), nx.complete_graph(3)),
}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, (name, G) in zip(axes, graphs.items()):
    nx.draw_networkx(G, pos=nx.spring_layout(G, seed=0), with_labels=True,
                     node_size=300, font_size=8, ax=ax)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# E1 TODO: implement 1-WL color refinement and visualize histogram evolution.
# Path5 and Star5 are same-size warmups. C6 and 2K3 are the same-size failure pair.

In [ ]:
# E2 TODO: show that constant-label 1-WL fails on C6 versus 2K3.
# These two graphs both have 6 nodes, so the failure is not caused by graph size.

In [ ]:
# E3 TODO: run RWSE on C6 versus 2K3 and explain the result.

In [ ]:
# E4 TODO: run node identity/coloring on C6 versus 2K3, visualize it, and explain why IDs are not permutation-invariant.

## Bonus: Oversquashing intuition on a barbell graph

In [ ]:
# Provided bonus graph visualization.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, G, title in zip(axes, [G_barbell, G_shortcut], ['Barbell', 'Barbell + shortcut']):
    nx.draw_networkx(G, pos=nx.spring_layout(G, seed=0), ax=ax,
                     node_size=150, with_labels=False, width=0.8)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Bonus TODO: report bottleneck statistics for G_barbell and G_shortcut.

In [ ]:
# Bonus TODO: inject signal at source_node=0, propagate with the symmetric GCN matrix, and plot signal reaching the right clique and bridge nodes.

In [ ]:
# Bonus TODO: plot variance and fixed-color signal snapshots for both graphs.